In [22]:
import pandas as pd
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed,ProcessPoolExecutor
from urllib.parse import quote_plus
from rdflib.namespace import RDF, DC, Namespace
import xml.etree.ElementTree as ET
from lxml import etree
import shutil
import zipfile
import ftplib
import io
import csv
import field_extractor2 as fe

# Constants
UNZIP_DIR = "selected_data"
FTP_HOST = "download.europeana.eu"
FTP_PATH = "dataset/XML/"
OUTPUT_DIR = "collected_data"

In [33]:
# Open and read the CSV file
with open('/home/sbasir/Thesis/Thesis/EDP/sample_data/datasets.csv', newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    
    # Extract 'ids' and 'lang' and save them in a dictionary
    data_dict = {row['ids'] + '.zip': row['lang'] for row in reader}

    # Reset the reader to start extracting only 'ids' in a separate list
    csvfile.seek(0)  # Rewind the CSV file
    next(reader)  # Skip the header
    
    # Extract only 'ids' in a separate list
    data_ids = [row['ids'] + '.zip' for row in reader]

# Print the dictionary to verify
print(data_dict)

data_ids = data_ids[14]
print(data_ids)

{'00101.zip': 'pt', '00718.zip': 'de', '00719.zip': 'de', '00720.zip': 'de', '00721.zip': 'de', '00722.zip': 'de', '00723.zip': 'de', '00724.zip': 'de', '00725.zip': 'de', '00732.zip': 'de', '00733.zip': 'de', '00735.zip': 'de', '00736.zip': 'de', '00737.zip': 'de', '00738.zip': 'de', '00739.zip': 'de', '00740.zip': 'de', '00741.zip': 'de', '00742.zip': 'de', '00743.zip': 'de', '00744.zip': 'de', '00745.zip': 'de', '00746.zip': 'de', '00747.zip': 'de', '00902.zip': 'de', '02030.zip': 'pt', '02301.zip': 'it', '02302.zip': 'it', '03706.zip': 'fr', '03707.zip': 'fr', '03709.zip': 'fr', '03710.zip': 'fr', '03915.zip': 'fr', '03916.zip': 'fr', '03919.zip': 'fr', '03924.zip': 'fr', '03927.zip': 'fr', '03928.zip': 'fr', '03929.zip': 'fr', '03930.zip': 'fr', '04802.zip': 'fr', '05813.zip': 'ro', '05815.zip': 'ro', '05816.zip': 'ro', '07101.zip': 'sk', '07931.zip': 'de', '07932.zip': 'de', '08001.zip': 'fr', '08520.zip': 'sl', '08534.zip': 'pl', '08535.zip': 'de', '08547.zip': 'de', '08604.zip'

In [14]:
# # Check if UNZIP_DIR exists, if it does, delete it
# if os.path.exists(UNZIP_DIR):
#     shutil.rmtree(UNZIP_DIR)

# # Check if OUTPUT_DIR exists, if it does, delete it
# if os.path.exists(OUTPUT_DIR):
#     shutil.rmtree(OUTPUT_DIR)

In [15]:
# Create the directories again
# os.makedirs(UNZIP_DIR)
# os.makedirs(OUTPUT_DIR)

In [34]:
import os
import random
import lxml.etree as ET

def download_file(ftp_host, ftp_path, filename):
    zip_data = io.BytesIO()

    with ftplib.FTP(ftp_host) as ftp:
        ftp.login()  # Login as anonymous
        ftp.cwd(ftp_path)

        ftp.retrbinary(f'RETR {filename}', zip_data.write)
    
    zip_data.seek(0)
    return zip_data

# Function to unzip a file
def unzip_file(zip_data):
    extracted_files = []  # List to store file content

    with zipfile.ZipFile(zip_data, 'r') as zip_ref:
        for file_info in zip_ref.infolist():
            with zip_ref.open(file_info) as file:
                file_content = file.read()
                extracted_files.append(file_content)

    return extracted_files

def keep_random_10_percent(files):
    # Calculate 10% of the total number of files
    num_files_to_keep = max(1, int(len(files) * 0.1))  # Ensure at least one file is kept
    print(f"keep {num_files_to_keep} documents")

    # Randomly sample 10% of the files to keep
    files_to_keep = random.sample(files, num_files_to_keep)

    return files_to_keep

def load_and_print_xml(file_path):
    # Parse the XML file
    tree = ET.parse(file_path)
    
    # Get the root element
    root = tree.getroot()
    
    return root

# Function to process a single subdirectory
def process_single_subdirectory(subdirectory_path):
    try:
        subdirectory = os.path.basename(subdirectory_path)
        print(f"Processing {subdirectory}...")

        # Get all files in the subdirectory
        files = os.listdir(subdirectory_path)
        size = len(files)
        print(f"Size of {subdirectory}: {size}")

        # Process RDF files in the sampled list
        output = fe.parse_rdf_files(subdirectory_path, subdirectory)
        
        # Write the output to an XML file
        output_file = f'/home/sbasir/Thesis/Thesis/EDP/collected_data/{subdirectory}'
        fe.write_data(output, output_file)

        print(f"Finished processing {subdirectory}. Output written to {output_file}")

        # Delete the subdirectory
        shutil.rmtree(subdirectory_path)
        print(f"Deleted subdirectory {subdirectory_path}")

    except Exception as e:
        print(f"Error processing {subdirectory}: {e}")

def process_zip_threaded(directory):
    subdirectories = [os.path.join(directory, subdir) for subdir in os.listdir(directory) if os.path.isdir(os.path.join(directory, subdir))]

    with ProcessPoolExecutor(max_workers=10) as executor:  # Experiment with the number of workers
        futures = {executor.submit(process_single_subdirectory, subdir): subdir for subdir in subdirectories}

        # Use tqdm to show progress as futures are completed
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing subdirectories"):
            try:
                future.result()
            except Exception as e:
                subdirectory = futures[future]
                print(f"Error processing subdirectory {subdirectory}: {e}")

In [ ]:
# Function to download and process a ZIP file
def download_parsed_data(filename):
    try:
        # print(f"Starting download and processing for {filename}...")

        # Construct the file path for the translation CSV
        translation_subdirectory = filename.replace(".zip", ".csv")
        csv_path = f'/home/sbasir/Thesis/Thesis/EDP/sample_data/translations/{translation_subdirectory}'
        
        # Check if the CSV file exists
        if not os.path.exists(csv_path):
            lang = data_dict.get(filename)
            if lang != "en":
                print(f"not considering {filename}")
                return
            
        # Download the ZIP file into memory
        zip_data = download_file(FTP_HOST, FTP_PATH, filename)

        # Unzip and process the file
        extracted_files = unzip_file(zip_data)
        print(f"deleting zip file {filename}")
        del zip_data 

        sample = keep_random_10_percent(extracted_files)
        del extracted_files

        # make a subdirectory for the dataset
        dataset_dir = os.path.join(UNZIP_DIR, filename)
        dataset_dir = dataset_dir[:-4]
        if not os.path.exists(dataset_dir):
            os.makedirs(dataset_dir)
        
        # save the extracted files in the subdirectory
        for i, file_content in enumerate(sample):
            with open(os.path.join(dataset_dir, f"{i}.xml"), 'wb') as file:
                file.write(file_content)

        print(f"Finished processing {filename}")
        del sample

    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [35]:
download_parsed_data(data_ids)

deleting zip file 00738.zip
keep 2633 documents only
Finished processing 00738.zip


In [36]:
# Use ProcessPoolExecutor for downloading and processing files in parallel
with ThreadPoolExecutor(max_workers=10) as executor:  # Reduce to avoid CPU or I/O contention
    futures = [executor.submit(download_zip, filename) for filename in data_ids]

    # Use tqdm to show progress as futures are completed
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing ZIP files"):
        # This will raise exceptions if any occurred during processing
        future.result()

not considering 0
not considering 0
not considering 7
not considering 3
not considering 8
not considering z
not considering i
not considering p


Processing ZIP files: 100%|██████████| 9/9 [00:00<00:00, 35.13it/s]

Error processing .: 550 Failed to open file.


In [37]:
import os

def get_directory_size_in_gb(directory):
    total_size = 0
    # Walk through the directory and its subdirectories
    for dirpath, dirnames, filenames in os.walk(directory):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)
            # Check if it's a file (sometimes there could be broken symlinks)
            if os.path.isfile(file_path):
                # Add the file's size to the total size
                total_size += os.path.getsize(file_path)
    
    # Convert bytes to gigabytes
    size_in_gb = total_size / (1024 ** 3)  # 1 GB = 1024^3 bytes
    return size_in_gb

# Example usage
directory = '/home/sbasir/Thesis/Thesis/EDP/selected_data/03915'
directory_size_gb = get_directory_size_in_gb(directory)
print(f"Directory size: {directory_size_gb:.10f} GB")


Directory size: 0.0000000000 GB


In [41]:
import os

def count_files(directory):
    # Initialize a counter
    file_count = 0
    
    # Iterate over all items in the directory
    for item in os.listdir(directory):
        # Construct the full path
        item_path = os.path.join(directory, item)
        
        # Check if it's a file (not a subdirectory)
        if os.path.isfile(item_path):
            file_count += 1
    
    return file_count

# Example usage
directory = '/home/sbasir/Thesis/Thesis/EDP/selected_data/00738'
num_files = count_files(directory)
print(f"Number of files in {directory}: {num_files}")

Number of files in /home/sbasir/Thesis/Thesis/EDP/selected_data/00738: 2633


In [42]:
process_zip_threaded('/home/sbasir/Thesis/Thesis/EDP/selected_data')

Processing subdirectories:   0%|          | 0/1 [00:00<?, ?it/s]

Processing 00738...
Size of 00738: 2633
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
5000 completed
Finished processing 00738. Output written to /home/sbasir/Thesis/Thesis/EDP/collected_data/00738
Deleted subdirectory /home/sbasir/Thesis/Thesis/EDP/selected_data/00738


Processing subdirectories: 100%|██████████| 1/1 [00:31<00:00, 31.07s/it]
